# 00 — Load and sanity

Resolve ``RUN_DIR`` under ``d-mercator-run/<RUN_SUBDIR>/``, load coordinates and edges, run basic checks, and write ``cache/merged.parquet`` for downstream notebooks.

**Resolve run directory**

Edit `RUN_SUBDIR` (e.g. `"d4"`) or set environment variable `DMERCATOR_RUN` before launching Jupyter so all paths point under `d-mercator-run/<name>/`. The printed `exists=` flags confirm Mercator output files are present.

In [11]:
import os
from pathlib import Path

import dmercator_io as dm

# Set default run subfolder here, or export DMERCATOR_RUN before Jupyter starts
RUN_SUBDIR = "d2"
RUN_SUBDIR = os.environ.get("DMERCATOR_RUN", RUN_SUBDIR)
RUN_DIR = dm.get_run_dir(RUN_SUBDIR)
paths = dm.paths_for_run(RUN_SUBDIR)
print("RUN_DIR:", RUN_DIR.resolve())
for k, v in paths.items():
    print(f"  {k}: {v} exists={v.is_file()}")


RUN_DIR: C:\Users\john\Documents\philipp\hbol\d-mercator-run\d2
  inf_coord: C:\Users\john\Documents\philipp\hbol\d-mercator-run\d2\edges_GC.inf_coord exists=True
  edge: C:\Users\john\Documents\philipp\hbol\d-mercator-run\d2\edges_GC.edge exists=True
  inf_log: C:\Users\john\Documents\philipp\hbol\d-mercator-run\d2\edges_GC.inf_log exists=True


**Load embeddings and graph, build merged table**

Parses `inf_coord` into a DataFrame, loads the undirected edge list, and adds a `degree` column from NetworkX. **Interpretation:** `coord rows` should match `graph |V|` and the header `nb_vertices`; `|E|` is the undirected edge count (each edge once).

In [12]:
meta, df = dm.parse_inf_coord(paths["inf_coord"])
G = dm.load_edges_graph(paths["edge"])

print("meta:", {k: meta[k] for k in ("nb_vertices", "beta", "dimension") if k in meta})
print("coord rows:", len(df), "graph |V|:", G.number_of_nodes(), "|E|:", G.number_of_edges())

deg = dict(G.degree())
merged = df.assign(degree=df["Vertex"].map(deg))
merged["degree"] = merged["degree"].fillna(0).astype(int)
merged.head()


meta: {'nb_vertices': 17090, 'beta': 2.02898, 'dimension': 2}
coord rows: 17090 graph |V|: 17090 |E|: 298866


,Vertex,Inf.Kappa,Inf.Hyp.Rad,Inf.Pos.1,Inf.Pos.2,Inf.Pos.3,degree
0,AR,731.5360,12.2905,10.96580,-22.6755,-26.9361,258
1,C2,13.6037,16.2753,7.44475,24.2101,26.8035,12
2,C3,170.4760,13.7471,-9.22702,-26.8317,23.5563,62
3,C5,61.4501,14.7674,14.42630,24.0795,23.9173,25
4,C6,12.6061,16.3515,-7.28107,-14.5304,33.1033,5


**Graph vs table overlap**

Compares node sets and flags edges whose endpoints are missing from the coordinate table. For a clean PPI export you expect **0** in all three printed counts; otherwise investigate ID formatting (e.g. extra spaces or different gene symbols between files).

In [13]:
# Vertices: graph vs coordinate table
Vg = set(G.nodes())
Vc = set(merged["Vertex"])
only_g = Vg - Vc
only_c = Vc - Vg
print("only in graph (no coord):", len(only_g))
print("only in coord (isolated / missing from G):", len(only_c))

bad_edges = [(u, v) for u, v in G.edges() if u not in Vc or v not in Vc]
print("edges with missing endpoint in coord:", len(bad_edges))
if bad_edges[:3]:
    print("sample:", bad_edges[:3])


only in graph (no coord): 0
only in coord (isolated / missing from G): 0
edges with missing endpoint in coord: 0


**Vertex ID consistency (summary)**

Reprints the symmetric mismatch size from the previous checks. Expect **0** when the graph and coordinate table use identical protein identifiers.

In [14]:
# Symmetric vertex mismatch (coord vs graph)
s = only_g | only_c
print("|symmetric vertex mismatch|:", len(s))
if s and len(s) <= 20:
    print(sorted(s))


|symmetric vertex mismatch|: 0


**Cache output**

Writes `cache/merged.parquet` (one row per protein with coordinates plus `degree`). Re-run this cell after you change `RUN_SUBDIR` so downstream notebooks and `viz_*` files read the same run.

In [15]:
cache_parquet = Path("cache") / "merged.parquet"
dm.save_merged_parquet(merged, cache_parquet)
print("Wrote", cache_parquet.resolve(), "rows:", len(merged))


Wrote C:\Users\john\Documents\philipp\hbol\notebooks\dmercator\cache\merged.parquet rows: 17090
